In [1]:
%load_ext autoreload
%autoreload 2

import numpy as np
import sys
sys.path.append('..')

from src.rl.sources import OUSource, ReplaySource
from src.rl.env import BatteryEnv

# Smoke test
source = OUSource(theta=0.05, mu=0, sigma=10)
f = np.ones(336) * 50
params = {"u_max": 25, "eta": 0.85, "S_max": 100, "S_0": 0, "dt": 1, "q": 50}

env = BatteryEnv(source, f, params, sigma_pred=10)
obs, info = env.reset()
print(f"Obs shape: {obs.shape}")
print(f"Obs: {obs}")

obs, reward, done, truncated, info = env.step(2)
print(f"After charge: SOC={info['soc']:.1f}, reward={reward:.4f}")

Obs shape: (17,)
Obs: [0.         0.46829128 0.         1.         0.         0.
 0.         0.         0.         0.         0.         1.
 1.         0.         0.32917953 0.         0.2853181 ]
After charge: SOC=21.2, reward=-0.7055


In [4]:
from src.rl.sources import ReplaySource
from src.simulation import walk_forward_backtest
from src.price_model import fit_seasonal_fourier, build_fourier_features
from src.optimization import get_optimal_policy, build_transition_matrix
from src.price_model import fit_seasonal, estimate_ou_params
from pathlib import Path
import pandas as pd

DATA_DIR = Path("../data")
ciso = pd.read_csv(DATA_DIR / "prices_CISO.csv")
ciso["hour"] = pd.to_datetime(ciso["hour"])
ciso["hour_of_day"] = ciso["hour"].dt.hour
ciso["dow"] = ciso["hour"].dt.dayofweek
ciso["month"] = ciso["hour"].dt.month

# Run one walk-forward iteration manually
train_window = 24 * 365
eval_window = 24 * 7
buffer = 24 * 7
X_grid = np.linspace(-100, 200, 200)
soc_grid = np.linspace(0, 100, 81)
params = {"u_max": 25, "eta": 0.85, "S_max": 100, "S_0": 0, "dt": 1}

start = 0
train_end = start + train_window
split = start + int(train_window * 0.75)

# Fit seasonal (inner)
seasonal_first = ciso.iloc[start:split].copy()
seasonal_model_inner, feature_cols_inner = fit_seasonal_fourier(seasonal_first)

# Pseudo-OOS residuals
validation = ciso.iloc[split:train_end].copy()
val_features = build_fourier_features(validation, feature_cols_inner)
val_resid = validation["price_usd_mwh"].values - seasonal_model_inner.predict(val_features).values
val_resid = val_resid - val_resid.mean()
theta, mu, sigma = estimate_ou_params(pd.Series(val_resid))
theta = max(theta, 0.01)

# Full window refit
full_train = ciso.iloc[start:train_end].copy()
seasonal_model, feature_cols = fit_seasonal_fourier(full_train)

# Mu fix
lookback = ciso.iloc[train_end - eval_window:train_end].copy()
lookback_features = build_fourier_features(lookback, feature_cols)
L_prev = (lookback["price_usd_mwh"].values
          - seasonal_model.predict(lookback_features).values).mean()
mu_eff = 0.6 * L_prev

trans = build_transition_matrix(theta, mu_eff, sigma, X_grid)

# Eval data
eval_data = ciso.iloc[train_end:train_end + eval_window + buffer].copy()
eval_features = build_fourier_features(eval_data, feature_cols)
f_eval = seasonal_model.predict(eval_features).values
X_eval_resid = eval_data["price_usd_mwh"].values - f_eval

params["q"] = full_train["price_usd_mwh"].mean()

# Solve DP
policy = get_optimal_policy(trans, f_eval, X_grid, soc_grid, params)

# --- DP via simulate() ---
from src.simulation import simulate
rev_dp, traj_dp = simulate(policy, f_eval, X_eval_resid, X_grid, soc_grid, params, T_eval=eval_window)

# --- DP via BatteryEnv + ReplaySource ---
source = ReplaySource(X_eval_resid[:eval_window + buffer], calib=(r["theta"],r["mu_eff"], r["sigma"]))
env = BatteryEnv(source, f_eval, params)

obs, info = env.reset(seed=0)
rev_env = 0
for t in range(eval_window + buffer):
    x_idx = np.argmin(np.abs(X_grid - env.X[t]))
    s_idx = np.argmin(np.abs(soc_grid - env.soc))
    
    action_u = policy[t, x_idx, s_idx]
    # Map continuous action to discrete
    if action_u > 0:
        action = 0   # discharge
    elif action_u < 0:
        action = 2   # charge
    else:
        action = 1   # hold
    
    obs, reward, done, truncated, info = env.step(action)
    if done:
        break

rev_env = info["revenue"]

print(f"DP simulate revenue:  ${rev_dp:,.2f}")
print(f"Env replay revenue:   ${rev_env:,.2f}")
print(f"Difference:           ${abs(rev_dp - rev_env):,.2f}")

NameError: name 'r' is not defined

In [3]:
from src.rl.sources import ReplaySource
from src.simulation import walk_forward_backtest
from src.price_model import fit_seasonal_fourier, build_fourier_features
from src.optimization import get_optimal_policy, build_transition_matrix
from src.price_model import fit_seasonal, estimate_ou_params
from pathlib import Path
import pandas as pd


ciso_results = walk_forward_backtest(ciso)


soc_grid = np.linspace(0, 100, 81)
params = {"u_max": 25, "eta": 0.85, "S_max": 100, "S_0": 0, "dt": 1}
train_window = 24 * 365

mismatches = 0
for r in ciso_results:
    policy = r["policy"]
    f_eval = r["f_eval"]
    X_eval_resid = r["X_eval_resid"]
    X_grid = r["X_grid"]

    start = r["eval_start"]
    train_data = ciso.iloc[start - train_window:start]
    params["q"] = train_data["price_usd_mwh"].mean()
    
    source = ReplaySource(X_eval_resid, calib=(r["theta"],r["mu_eff"], r["sigma"]))
    env = BatteryEnv(source, f_eval, params)
    obs, info = env.reset(seed=0)
    
    for t in range(len(f_eval)):
        x_idx = np.argmin(np.abs(X_grid - env.X[t]))
        s_idx = np.argmin(np.abs(soc_grid - env.soc))
        action_u = policy[t, x_idx, s_idx]
        
        if action_u > 0:
            action = 0
        elif action_u < 0:
            action = 2
        else:
            action = 1
        
        obs, reward, done, truncated, info = env.step(action)
        if done:
            break
    
    diff = abs(r["revenue"] - info["revenue"])
    if diff > 0.01:
        mismatches += 1
        print(f"Week {r['eval_start']}: DP=${r['revenue']:,.2f}, Env=${info['revenue']:,.2f}, diff=${diff:,.2f}")

print(f"\n{mismatches} mismatches out of {len(ciso_results)} weeks")

Week  1 | rev = $    16,235 | theta = 0.0213 | sigma=7.88 | mu=-14.57
Week  2 | rev = $    10,319 | theta = 0.0178 | sigma=7.72 | mu=-14.56
Week  3 | rev = $    15,746 | theta = 0.0206 | sigma=7.52 | mu=-13.28
Week  4 | rev = $    21,949 | theta = 0.0198 | sigma=7.29 | mu=-8.33
Week  5 | rev = $    19,593 | theta = 0.0190 | sigma=7.17 | mu=-8.27
Week  6 | rev = $    18,423 | theta = 0.0160 | sigma=7.00 | mu=-7.69
Week  7 | rev = $    10,708 | theta = 0.0123 | sigma=6.91 | mu=-4.13
Week  8 | rev = $    18,364 | theta = 0.0137 | sigma=6.81 | mu=-6.29
Week  9 | rev = $    25,163 | theta = 0.0203 | sigma=6.29 | mu=-6.65
Week 10 | rev = $    24,377 | theta = 0.0177 | sigma=5.66 | mu=-7.91
Week 11 | rev = $    19,088 | theta = 0.0100 | sigma=6.45 | mu=-7.28
Week 12 | rev = $    23,249 | theta = 0.0153 | sigma=5.80 | mu=1.64
Week 13 | rev = $    20,677 | theta = 0.0167 | sigma=5.81 | mu=2.26
Week 14 | rev = $    26,022 | theta = 0.0157 | sigma=5.88 | mu=-1.48
Week 15 | rev = $    18,089 | the

In [4]:
from src.rl.sources import OUSource

n_episodes = 1000
gate2_revenues = []

# Use a representative week's parameters and seasonal forecast
r = ciso_results[len(ciso_results) // 2]  # middle week
f_eval = r["f_eval"]
X_grid = r["X_grid"]
policy = r["policy"]
theta = r["theta"]
sigma = r["sigma"]
mu_eff = r["mu_eff"]

start = r["eval_start"]
train_data = ciso.iloc[start - train_window:start]
params["q"] = train_data["price_usd_mwh"].mean()

source = OUSource(theta=theta, mu=mu_eff, sigma=sigma)

for ep in range(n_episodes):
    env = BatteryEnv(source, f_eval, params)
    obs, info = env.reset(seed=ep)
    
    for t in range(len(f_eval)):
        x_idx = np.argmin(np.abs(X_grid - env.X[t]))
        s_idx = np.argmin(np.abs(soc_grid - env.soc))
        action_u = policy[t, x_idx, s_idx]
        
        if action_u > 0:
            action = 0
        elif action_u < 0:
            action = 2
        else:
            action = 1
        
        obs, reward, done, truncated, info = env.step(action)
        if done:
            break
    
    gate2_revenues.append(info["revenue"])

print(f"Gate 2 — DP on OUSource ({n_episodes} episodes)")
print(f"  Mean revenue:   ${np.mean(gate2_revenues):,.0f}")
print(f"  Std revenue:    ${np.std(gate2_revenues):,.0f}")
print(f"  Min revenue:    ${np.min(gate2_revenues):,.0f}")
print(f"  Max revenue:    ${np.max(gate2_revenues):,.0f}")
print(f"  95th pct floor: ${np.percentile(gate2_revenues, 5):,.0f}")

Gate 2 — DP on OUSource (1000 episodes)
  Mean revenue:   $18,982
  Std revenue:    $3,045
  Min revenue:    $9,699
  Max revenue:    $30,414
  95th pct floor: $13,981


In [5]:
from src.rl.sources import OUSource, sample_f

source = OUSource(theta=0.05, mu=0, sigma=10)
params["q"] = 50
env = BatteryEnv(source, np.zeros(336), params, f_sampler=sample_f)

# Each reset should produce a different seasonal curve
for i in range(3):
    obs, info = env.reset(seed=i)
    print(f"Reset {i}: f mean={env.f.mean():.1f}, f std={env.f.std():.1f}")

Reset 0: f mean=54.1, f std=4.8
Reset 1: f mean=50.4, f std=37.8
Reset 2: f mean=42.8, f std=5.2


In [ ]:
from src.rl.train import make_vec_env, Gate2EvalCallback
from sb3_contrib import MaskablePPO

# After Gate 1 passes, strip heavy artifacts
for r in ciso_results:
    del r["policy"]
    del r["f_eval"]
    del r["X_eval_resid"]
    del r["X_grid"]

import gc
gc.collect()


# Training env
params["q"] = ciso["price_usd_mwh"].mean()
venv = make_vec_env(params, n_envs=8)

# Gate 2 calibration from middle week
r = ciso_results[len(ciso_results) // 2]
start = r["eval_start"]
train_data = ciso.iloc[start - train_window:start]
params["q"] = train_data["price_usd_mwh"].mean()

callback = Gate2EvalCallback(
    f_eval=r["f_eval"],
    params=params,
    theta=r["theta"],
    mu=r["mu_eff"],
    sigma=r["sigma"],
    n_eval_episodes=200,
    eval_freq=10_000,
)

model = MaskablePPO(
    "MlpPolicy", venv,
    gamma=1.0,
    n_steps=1008,
    batch_size=512,
    learning_rate=3e-4,
    n_epochs=10,
    gae_lambda=0.95,
    ent_coef=0.01,
    policy_kwargs=dict(net_arch=[128, 128]),
    tensorboard_log="runs/",
    seed=0,
    verbose=0,
)

model.learn(total_timesteps=50_000, callback=callback)
venv.close()

In [1]:
import psutil
print(f"Available: {psutil.virtual_memory().available / 1e9:.1f} GB")
print(f"Total:     {psutil.virtual_memory().total / 1e9:.1f} GB")

Available: 0.7 GB
Total:     8.6 GB


In [1]:
import gc

# Delete stored policies if Gate 1 already passed
if 'ciso_results' in dir():
    for r in ciso_results:
        for key in ["policy", "f_eval", "X_eval_resid", "X_grid"]:
            if key in r:
                del r[key]

# Delete any large arrays from earlier cells
for var in ['policy', 'trans', 'V', 'traj_dp', 'traj_sim']:
    if var in dir():
        exec(f"del {var}")

gc.collect()

19

In [ ]:
import sys
sys.path.append('..')
import numpy as np
import pandas as pd
from pathlib import Path
from src.rl.train import make_vec_env, Gate2EvalCallback
from src.rl.sources import OUSource
from src.price_model import fit_seasonal_fourier, build_fourier_features
from sb3_contrib import MaskablePPO

# Load just enough data to get Gate 2 parameters
DATA_DIR = Path("../data")
ciso = pd.read_csv(DATA_DIR / "prices_CISO.csv")
ciso["hour"] = pd.to_datetime(ciso["hour"])
ciso["hour_of_day"] = ciso["hour"].dt.hour
ciso["dow"] = ciso["hour"].dt.dayofweek
ciso["month"] = ciso["hour"].dt.month

# Middle week parameters — hardcode from your Gate 2 results
# rather than rerunning walk_forward_backtest
train_window = 24 * 365
eval_window = 24 * 7
mid_start = len(ciso) // 2
train_end = mid_start

full_train = ciso.iloc[mid_start - train_window:mid_start].copy()
seasonal_model, feature_cols = fit_seasonal_fourier(full_train)

eval_data = ciso.iloc[mid_start:mid_start + eval_window + eval_window].copy()
eval_features = build_fourier_features(eval_data, feature_cols)
f_eval = seasonal_model.predict(eval_features).values

# Hardcode from your previous Gate 2 / backtest output
theta = 0.0454  # middle week theta
mu_eff = -1.5   # middle week mu_eff  
sigma = 6.34    # middle week sigma
q = full_train["price_usd_mwh"].mean()

params = {"u_max": 25, "eta": 0.85, "S_max": 100, "S_0": 0, "dt": 1, "q": q}

# Drop the full dataframe — don't need it anymore
del ciso, full_train, eval_data, eval_features, seasonal_model
import gc
gc.collect()

print(f"Available RAM: {__import__('psutil').virtual_memory().available / 1e9:.1f} GB")

# Train with 4 envs to save memory
venv = make_vec_env(params, n_envs=4)

callback = Gate2EvalCallback(
    f_eval=f_eval,
    params=params,
    theta=theta,
    mu=mu_eff,
    sigma=sigma,
    n_eval_episodes=100,  # reduced from 200
    eval_freq=10_000,
)

model = MaskablePPO(
    "MlpPolicy", venv,
    gamma=1.0,
    n_steps=1008,
    batch_size=504,
    learning_rate=3e-4,
    n_epochs=10,
    gae_lambda=0.95,
    ent_coef=0.01,
    policy_kwargs=dict(net_arch=[64, 64]),  # smaller network
    tensorboard_log="runs/",
    seed=0,
    verbose=0,
)

model.learn(total_timesteps=50_000, callback=callback)
venv.close()

In [1]:
import sys
sys.path.append('..')
import numpy as np
import pandas as pd
from pathlib import Path
import gc

# Check memory before anything
import psutil
print(f"Available before loading: {psutil.virtual_memory().available / 1e9:.1f} GB")

# Load minimal data
DATA_DIR = Path("../data")
ciso = pd.read_csv(DATA_DIR / "prices_CISO.csv")
ciso["hour"] = pd.to_datetime(ciso["hour"])
ciso["hour_of_day"] = ciso["hour"].dt.hour
ciso["dow"] = ciso["hour"].dt.dayofweek
ciso["month"] = ciso["hour"].dt.month

# Extract only what training needs
from src.price_model import fit_seasonal_fourier, build_fourier_features

train_window = 24 * 365
eval_window = 24 * 7
mid = len(ciso) // 2

full_train = ciso.iloc[mid - train_window:mid].copy()
seasonal_model, feature_cols = fit_seasonal_fourier(full_train)

eval_data = ciso.iloc[mid:mid + eval_window * 2].copy()
eval_features = build_fourier_features(eval_data, feature_cols)
f_eval = seasonal_model.predict(eval_features).values

q = full_train["price_usd_mwh"].mean()

# Delete everything heavy
del ciso, full_train, eval_data, eval_features, seasonal_model, feature_cols
gc.collect()

print(f"Available after cleanup: {psutil.virtual_memory().available / 1e9:.1f} GB")

Available before loading: 0.6 GB
Available after cleanup: 0.6 GB
